# SportsMOT basketball subset -> Drive (one-time prep)
Runs in Colab (CPU runtime is enough). It downloads the official SportsMOT `train.tar` and `val.tar`
from Hugging Face (`MCG-NJU/SportsMOT`) to the Colab local disk, checks their SHA256, and keeps only the
basketball sequences (15 train + 15 val). The frames are stored as the original JPGs, not re-encoded.
Each split is written as one `.tar` to `MyDrive/TacticalAI/datasets/sportsmot/`.

* License: **CC BY-NC 4.0**. Internal, non-commercial evaluation only. Do not redistribute or commit to git.
* Ground truth (GT) covers **on-court players only**, not referees or coaches. The clips have no shot cuts.
* The test split has no public GT and is not downloaded.
* Peak local disk use is about 9 GB, because the splits are processed one at a time.
* After this, use the last cell in each new session to load the data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import hashlib, io, json, shutil, subprocess, tarfile, time, urllib.request
from pathlib import Path

DRIVE_OUT = Path('/content/drive/MyDrive/TacticalAI/datasets/sportsmot')
WORK = Path('/content/sportsmot_work')
HF = 'https://huggingface.co/datasets/MCG-NJU/SportsMOT/resolve/main'
EXPECTED = {
    'val': (6_592_876_544, '35f1c585297426a2ec104f645c6bf8684e7e856c536775b167fe3f2ffd388ac3'),
    'train': (6_703_191_552, 'dc8edc8126bcb66611e768679f618a14880510944dc0e07959c00c0c14722341'),
}
VERSION = 'v1'
for d in (DRIVE_OUT, WORK):
    d.mkdir(parents=True, exist_ok=True)

def fetch_text(path):
    return urllib.request.urlopen(f'{HF}/{path}').read().decode()

splits = {n: fetch_text(f'splits_txt/{n}.txt') for n in ('basketball', 'train', 'val')}
seqs = lambda n: {s.strip() for s in splits[n].splitlines() if s.strip()}
basketball = seqs('basketball')
print({s: len(basketball & seqs(s)) for s in ('train', 'val')}, 'basketball sequences per split')
print(shutil.disk_usage('/content'))

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 24), b''):
            h.update(chunk)
    return h.hexdigest()

def add_bytes(tar, name, data):
    info = tarfile.TarInfo(name)
    info.size, info.mtime, info.mode = len(data), int(time.time()), 0o644
    tar.addfile(info, io.BytesIO(data))

def prepare(split):
    size, oid = EXPECTED[split]
    final = DRIVE_OUT / f'sportsmot_basketball_{split}_{VERSION}.tar'
    if final.exists():
        print(f'{split}: already on Drive -> {final}')
        return
    src = WORK / f'{split}.tar'
    subprocess.run(['curl', '-L', '-C', '-', '--retry', '5', '--fail', '-sS', '-o', str(src),
                    f'{HF}/dataset/{split}.tar'], check=True)
    assert src.stat().st_size == size, f'{split}: size {src.stat().st_size} != {size}'
    assert sha256(src) == oid, f'{split}: sha256 mismatch'
    print(f'{split}: download verified')

    wanted = basketball & seqs(split)
    prefix = f'sportsmot_basketball/{split}'
    local = WORK / final.name
    frames, gt_rows, found = {}, {}, set()
    with tarfile.open(src, 'r|') as tin, tarfile.open(local, 'w') as tout:
        for m in tin:
            parts = Path(m.name).parts
            seq = next((p for p in parts if p in wanted), None)
            if seq is None or not m.isfile():
                continue
            rel = '/'.join(parts[parts.index(seq):])
            data = tin.extractfile(m).read()
            info = tarfile.TarInfo(f'{prefix}/{rel}')
            info.size, info.mtime, info.mode = len(data), m.mtime, 0o644
            tout.addfile(info, io.BytesIO(data))
            found.add(seq)
            if parts[-2] == 'img1':
                frames[seq] = frames.get(seq, 0) + 1
            elif rel.endswith('gt/gt.txt'):
                gt_rows[seq] = sum(1 for line in data.splitlines() if line.strip())
        missing = sorted(wanted - found)
        assert not missing, f'{split}: missing {missing}'
        manifest = {
            'dataset': 'SportsMOT basketball subset', 'version': VERSION, 'split': split,
            'source': 'https://huggingface.co/datasets/MCG-NJU/SportsMOT',
            'source_file': f'dataset/{split}.tar', 'source_sha256': oid,
            'license': 'CC BY-NC 4.0 - non-commercial, no redistribution',
            'annotations': 'MOT format, on-court players only (no referees/coaches), 1280x720 @ 25 fps',
            'sequences': {s: {'frames': frames.get(s, 0), 'gt_rows': gt_rows.get(s)} for s in sorted(found)},
            'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        }
        blob = json.dumps(manifest, indent=2).encode()
        add_bytes(tout, f'{prefix}/manifest.json', blob)
        for name in ('basketball', split):
            add_bytes(tout, f'{prefix}/splits/{name}.txt', splits[name].encode())
    src.unlink()
    manifest['archive_sha256'] = sha256(local)
    tmp = final.with_suffix('.tar.partial')
    shutil.copyfile(local, tmp)
    tmp.rename(final)
    final.with_suffix('.manifest.json').write_text(json.dumps(manifest, indent=2))
    local.unlink()
    print(f'{split}: {len(found)} seqs, {sum(frames.values())} frames, '
          f'{final.stat().st_size / 1e9:.2f} GB -> {final}')

for split in ('val', 'train'):
    prepare(split)

In [ ]:
for p in sorted(DRIVE_OUT.iterdir()):
    print(f'{p.stat().st_size / 1e9:8.2f} GB  {p.name}')
print(json.dumps(json.loads((DRIVE_OUT / f'sportsmot_basketball_val_{VERSION}.manifest.json').read_text())['sequences'], indent=1))

## Per-session loader (copy into experiment notebooks)
This copies the large archives from Drive to the local disk and extracts them there. It gives
`/content/data/sportsmot_basketball/{train,val}/<seq>/{img1,gt,seqinfo.ini}`.

In [ ]:
import shutil, subprocess
from pathlib import Path
SRC = Path('/content/drive/MyDrive/TacticalAI/datasets/sportsmot')
DST = Path('/content/data')
DST.mkdir(parents=True, exist_ok=True)
for split in ('val', 'train'):
    if (DST / 'sportsmot_basketball' / split).is_dir():
        continue
    local = DST / f'sportsmot_basketball_{split}_v1.tar'
    shutil.copyfile(SRC / local.name, local)
    subprocess.run(['tar', '-xf', str(local), '-C', str(DST)], check=True)
    local.unlink()
print(sorted(p.name for p in (DST / 'sportsmot_basketball' / 'val').iterdir()))